In [1]:
# =============================================================================
# 12c — Overlap metrics
#
# Builds one long table: image x target x CAM source x metric.
# No statistics, no plots. Those live in 12d.
#
# Scoring happens at the native 14x14 CAM grid. Saved CAMs are 224x224
# but were upsampled from 14x14, so the extra resolution is interpolation.
# =============================================================================
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score

def find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "generate_finer_cam_panderm.py").exists():
            return p.resolve()
    raise FileNotFoundError

REPO = find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.eval.cam_eval_utils import (
    resize_centercrop_mask, mask_to_cam_grid, cam_to_grid, norm01, CAM_GRID)

EVAL_ROOT = REPO / "outputs" / "mel_nv" / "eval_cams"
QC_CSV    = REPO / "results" / "annotation_qc" / "annotation_qc_manifest.csv"
OUT_DIR   = REPO / "results" / "overlap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K       = 20      # fixed patch count, ~10 percent of 196
ANN_THRESH  = 0.5     # patch counts as annotated if at least half covered
N_PERM      = 2000

qc   = pd.read_csv(QC_CSV)
qcp  = qc[qc.qc_pass].copy()
pred = pd.read_csv(EVAL_ROOT / "predictions.csv")
man  = pd.read_csv(EVAL_ROOT / "cam_manifest.csv")

print(f"annotations passing QC : {len(qcp)}")
print(f"images                 : {qcp.Image_ID.nunique()}")
print(f"CAM arrays             : {len(man)}")
print(f"top-k                  : {TOP_K} of {CAM_GRID**2} patches "
      f"({TOP_K/CAM_GRID**2:.1%})")

annotations passing QC : 102
images                 : 34
CAM arrays             : 850
top-k                  : 20 of 196 patches (10.2%)


In [2]:
# =============================================================================
# Targets, all at 14x14 binary.
#   group unions  MEL, NV, COARSE   -> E1 and E2
#   single labels                   -> E3, descriptive
# =============================================================================
def union_grid(sub):
    acc = None
    for _, r in sub.iterrows():
        m = np.array(Image.open(r.mask_path).convert("L")) > 127
        acc = m if acc is None else (acc | m)
    crop, _ = resize_centercrop_mask(acc)
    return mask_to_cam_grid(crop, already_cropped=True) >= ANN_THRESH

targets = {}   # (image_id, target_name) -> bool 14x14

for (img, grp), sub in qcp.groupby(["Image_ID", "label_group"]):
    targets[(img, f"union_{grp}")] = union_grid(sub)

for (img, lab), sub in qcp.groupby(["Image_ID", "label"]):
    targets[(img, f"label_{lab}")] = union_grid(sub)

tinfo = pd.DataFrame([
    {"image_id": k[0], "target": k[1],
     "n_patches": int(v.sum()),
     "area_frac": float(v.mean())}
    for k, v in targets.items()
])
tinfo.to_csv(OUT_DIR / "target_summary.csv", index=False)

print(f"targets built: {len(targets)}")
print()
print(tinfo[tinfo.target.str.startswith("union_")]
      .groupby("target")[["n_patches", "area_frac"]]
      .agg(["count", "median", "min", "max"]).round(3).to_string())

degenerate = tinfo[(tinfo.n_patches == 0) | (tinfo.area_frac > 0.9)]
print(f"\ndegenerate targets, empty or near-full: {len(degenerate)}")
if len(degenerate):
    print(degenerate.to_string(index=False))

targets built: 147

             n_patches                 area_frac                     
                 count median min  max     count median    min    max
target                                                               
union_COARSE         3   17.0   9   34         3  0.087  0.046  0.173
union_MEL           31   44.0   7  111        31  0.224  0.036  0.566
union_NV             6   39.0  34  101         6  0.199  0.173  0.515
union_OTHER          5   13.0   3   46         5  0.066  0.015  0.235

degenerate targets, empty or near-full: 0


In [3]:
# =============================================================================
# hit_rate   share of top-k CAM MASS inside the target
# lift       hit_rate / area_frac. 1.0 is chance.
# pointing   hottest patch inside the target, 0 or 1
# cam_auc    threshold free ranking quality. 0.5 is chance. PRIMARY.
# recall_k   share of target covered by top-k. diagnostic only.
# =============================================================================
def topk_fixed(cam, k=TOP_K):
    flat = cam.ravel()
    idx = np.argpartition(-flat, k - 1)[:k]
    out = np.zeros(flat.size, bool)
    out[idx] = True
    return out.reshape(cam.shape)


def overlap_metrics(cam_grid, target, k=TOP_K):
    cam = norm01(cam_grid)
    t = np.asarray(target, bool)
    n = t.size
    out = {"area_frac": float(t.mean()), "n_target": int(t.sum())}

    if t.sum() == 0 or t.all():
        out.update({m: np.nan for m in
                    ["hit_rate", "lift", "pointing", "cam_auc", "recall_k"]})
        return out

    sel = topk_fixed(cam, k)
    mass = cam[sel].sum()
    out["hit_rate"] = float(cam[sel & t].sum() / (mass + 1e-8))
    out["lift"]     = out["hit_rate"] / (out["area_frac"] + 1e-8)
    out["pointing"] = float(t[np.unravel_index(np.argmax(cam), cam.shape)])
    out["cam_auc"]  = float(roc_auc_score(t.ravel(), cam.ravel()))
    out["recall_k"] = float((sel & t).sum() / t.sum())
    return out

In [6]:
# =============================================================================
# gradcam_a, gradcam_b, map_diff are identical across gamma for a given
# checkpoint, since gamma only affects FinerCAM. Keep gamma=0.6 for those
# and both gammas for finercam.
# =============================================================================
cams = {}      # (image_id, source) -> 14x14 float
src_meta = {}  # source -> dict of fields

for _, r in man.iterrows():
    if r.family == "activation":
        if r.cam_type != "finercam" and float(r.gamma) != 0.6:
            continue
        gtag = str(r.gamma).replace(".", "p")
        src = (f"act_{r.ckpt}_{r.cam_type}_gam{gtag}"
               if r.cam_type == "finercam"
               else f"act_{r.ckpt}_{r.cam_type}")
        gamma = float(r.gamma) if r.cam_type == "finercam" else np.nan
    else:
        src = f"attn_{r.ckpt}_{r.cam_type}"
        gamma = np.nan

    arr = np.load(REPO / r.path)
    cams[(r.image_id, src)] = (
        cam_to_grid(arr) if arr.shape != (CAM_GRID, CAM_GRID) else arr)

    src_meta[src] = {
        "family":   "attn" if r.family == "attention" else "act",
        "ckpt":     r.ckpt,
        "cam_type": r.cam_type,
        "gamma":    gamma,
    }

sources = sorted(src_meta)
print(f"loaded {len(cams)} arrays across {len(sources)} sources\n")
print(pd.DataFrame(src_meta).T.to_string())

loaded 748 arrays across 22 sources

                        family ckpt      cam_type gamma
act_ha0_finercam_gam0p6    act  ha0      finercam   0.6
act_ha0_gradcam_a          act  ha0     gradcam_a   NaN
act_ha0_gradcam_b          act  ha0     gradcam_b   NaN
act_ha0_map_diff           act  ha0      map_diff   NaN
act_ha3_finercam_gam0p6    act  ha3      finercam   0.6
act_ha3_gradcam_a          act  ha3     gradcam_a   NaN
act_ha3_gradcam_b          act  ha3     gradcam_b   NaN
act_ha3_map_diff           act  ha3      map_diff   NaN
act_ha5_finercam_gam0p6    act  ha5      finercam   0.6
act_ha5_gradcam_a          act  ha5     gradcam_a   NaN
act_ha5_gradcam_b          act  ha5     gradcam_b   NaN
act_ha5_map_diff           act  ha5      map_diff   NaN
act_ha5_finercam_gam0p8    act  ha5      finercam   0.8
attn_ha0_attncam_a        attn  ha0     attncam_a   NaN
attn_ha0_attncam_b        attn  ha0     attncam_b   NaN
attn_ha0_attncam_diff     attn  ha0  attncam_diff   NaN
attn_ha3_at

In [7]:
gt   = pred.set_index("image_id")["gt_label"].to_dict()
case = pred.set_index("image_id")["case_type"].to_dict()
conf = pred.set_index("image_id")["confidence"].to_dict()
corr = pred.set_index("image_id")["correct"].to_dict()

rows = []
for (img, tname), tmask in targets.items():
    for src in sources:
        cam = cams.get((img, src))
        if cam is None:
            continue
        rows.append({
            "image_id": img,
            "target": tname,
            "target_kind": "union" if tname.startswith("union_") else "label",
            "source": src,
            **src_meta[src],
            "gt_label": gt.get(img),
            "case_type": case.get(img),
            "confidence": conf.get(img),
            "correct": corr.get(img),
            **overlap_metrics(cam, tmask),
        })

long = pd.DataFrame(rows)
long.to_csv(OUT_DIR / "overlap_metrics_long.csv", index=False)

print(f"rows: {len(long)}")
print(f"nan cam_auc: {int(long.cam_auc.isna().sum())}")
print()
print("MEL union, cam_auc by source:")
print(long[long.target == "union_MEL"]
      .groupby(["family", "ckpt", "cam_type", "gamma"], dropna=False)["cam_auc"]
      .agg(["count", "median"]).round(3).to_string())

rows: 3234
nan cam_auc: 0

MEL union, cam_auc by source:
                                count  median
family ckpt cam_type     gamma               
act    ha0  finercam     0.6       31   0.575
            gradcam_a    NaN       31   0.555
            gradcam_b    NaN       31   0.400
            map_diff     NaN       31   0.583
       ha3  finercam     0.6       31   0.955
            gradcam_a    NaN       31   0.957
            gradcam_b    NaN       31   0.949
            map_diff     NaN       31   0.777
       ha5  finercam     0.6       31   0.956
                         0.8       31   0.911
            gradcam_a    NaN       31   0.955
            gradcam_b    NaN       31   0.956
            map_diff     NaN       31   0.720
attn   ha0  attncam_a    NaN       31   0.507
            attncam_b    NaN       31   0.475
            attncam_diff NaN       31   0.515
       ha3  attncam_a    NaN       31   0.923
            attncam_b    NaN       31   0.915
            attncam_dif

In [8]:
# =============================================================================
# Score each CAM against OTHER images' targets. Gives an empirical null
# that respects real CAM shape statistics, unlike the area baseline.
# =============================================================================
rng = np.random.default_rng(0)
null_rows = []

for tname in ["union_MEL", "union_NV", "union_COARSE"]:
    ids = [i for (i, t) in targets if t == tname]
    if len(ids) < 4:
        continue
    for src in sources:
        vals_auc, vals_lift = [], []
        for _ in range(N_PERM // 4):
            i, j = rng.choice(len(ids), 2, replace=False)
            cam = cams.get((ids[i], src))
            if cam is None:
                continue
            m = overlap_metrics(cam, targets[(ids[j], tname)])
            if not np.isnan(m["cam_auc"]):
                vals_auc.append(m["cam_auc"])
                vals_lift.append(m["lift"])
        if vals_auc:
            null_rows.append({
                "target": tname, "source": src,
                "null_auc_mean": float(np.mean(vals_auc)),
                "null_auc_p95": float(np.percentile(vals_auc, 95)),
                "null_lift_mean": float(np.mean(vals_lift)),
                "null_lift_p95": float(np.percentile(vals_lift, 95)),
            })

null = pd.DataFrame(null_rows)
null.to_csv(OUT_DIR / "permutation_null.csv", index=False)

obs = (long[long.target_kind == "union"]
       .groupby(["target", "source"])[["cam_auc", "lift"]]
       .median().reset_index()
       .rename(columns={"cam_auc": "obs_auc", "lift": "obs_lift"}))
cmp_ = obs.merge(null, on=["target", "source"], how="left")
cmp_["auc_above_null"]  = cmp_.obs_auc - cmp_.null_auc_mean
cmp_["beats_null_p95"]  = cmp_.obs_auc > cmp_.null_auc_p95
cmp_.to_csv(OUT_DIR / "observed_vs_null.csv", index=False)

print("MEL union, observed versus null:")
print(cmp_[cmp_.target == "union_MEL"]
      .sort_values("obs_auc", ascending=False)
      [["source", "obs_auc", "null_auc_mean", "auc_above_null", "beats_null_p95"]]
      .round(3).to_string(index=False))

MEL union, observed versus null:
                 source  obs_auc  null_auc_mean  auc_above_null  beats_null_p95
      act_ha3_gradcam_a    0.957          0.877           0.080           False
act_ha5_finercam_gam0p6    0.956          0.862           0.094           False
      act_ha5_gradcam_b    0.956          0.878           0.078           False
act_ha3_finercam_gam0p6    0.955          0.852           0.103           False
      act_ha5_gradcam_a    0.955          0.872           0.083           False
      act_ha3_gradcam_b    0.949          0.877           0.072           False
     attn_ha3_attncam_a    0.923          0.784           0.139            True
     attn_ha5_attncam_b    0.923          0.793           0.129            True
     attn_ha5_attncam_a    0.920          0.795           0.125           False
     attn_ha3_attncam_b    0.915          0.797           0.117           False
act_ha5_finercam_gam0p8    0.911          0.773           0.137           False
       

In [10]:
# =============================================================================
# Cell 6b — lesion-conditioned metrics
#
# Restrict the CAM and the target to lesion patches only. This removes the
# trivial advantage of firing inside a large lesion where all annotations
# happen to live.
#
# READ: AUC near 0.5 here means HA learned lesion localisation, not feature
#       localisation. AUC well above 0.5 means genuine within-lesion signal.
# =============================================================================
eval_df = pd.read_csv(EVAL_ROOT / "eval_images.csv")
HAM_ROOT = REPO / "data" / "HAM10000"
from src.eval.cam_eval_utils import mask_to_cam_grid, resize_centercrop_mask

lesion_grid = {}
for _, r in eval_df.iterrows():
    m = np.array(Image.open((REPO / "data" / "HAM10000" / str(r.mask_rel_path))
                            .resolve()).convert("L")) > 127
    lesion_grid[r.image_id] = mask_to_cam_grid(
        resize_centercrop_mask(m)[0], already_cropped=True) >= 0.5

print("lesion coverage at 14x14:")
la = pd.Series({k: v.mean() for k, v in lesion_grid.items()})
print(la.describe().round(3).to_string())


def conditioned_metrics(cam, target, lesion, k_frac=0.10):
    """Score only inside the lesion. k scales with lesion size."""
    inside = np.asarray(lesion, bool)
    t = np.asarray(target, bool) & inside
    n_in = int(inside.sum())

    if n_in < 10 or t.sum() == 0 or t.sum() == n_in:
        return {"c_auc": np.nan, "c_lift": np.nan, "c_area": np.nan}

    c = norm01(cam)[inside]
    tt = t[inside]
    k = max(1, int(round(k_frac * n_in)))
    sel = np.zeros(n_in, bool)
    sel[np.argpartition(-c, k - 1)[:k]] = True

    area = float(tt.mean())
    hit = float(c[sel & tt].sum() / (c[sel].sum() + 1e-8))
    return {
        "c_auc": float(roc_auc_score(tt, c)),
        "c_lift": hit / (area + 1e-8),
        "c_area": area,
    }


rows = []
for (img, tname), tmask in targets.items():
    if not tname.startswith("union_"):
        continue
    les = lesion_grid.get(img)
    if les is None:
        continue
    for src in sources:
        cam = cams.get((img, src))
        if cam is None:
            continue
        rows.append({"image_id": img, "target": tname, "source": src,
                     **src_meta[src], "gt_label": gt.get(img),
                     "case_type": case.get(img),
                     **conditioned_metrics(cam, tmask, les)})

cond = pd.DataFrame(rows)
cond.to_csv(OUT_DIR / "lesion_conditioned_metrics.csv", index=False)

print("\n\nMEL union, lesion-conditioned AUC:")
print(cond[cond.target == "union_MEL"]
      .groupby(["family", "ckpt", "cam_type", "gamma"], dropna=False)["c_auc"]
      .agg(["count", "median"]).round(3).to_string())

print("\n\nside by side, target-class CAM only:")
raw = (long[(long.target == "union_MEL") &
            (long.cam_type.isin(["gradcam_a", "attncam_a"]))]
       .groupby(["family", "ckpt"])["cam_auc"].median())
con = (cond[(cond.target == "union_MEL") &
            (cond.cam_type.isin(["gradcam_a", "attncam_a"]))]
       .groupby(["family", "ckpt"])["c_auc"].median())
print(pd.DataFrame({"raw_auc": raw, "lesion_conditioned_auc": con})
      .round(3).to_string())

lesion coverage at 14x14:
count    34.000
mean      0.468
std       0.207
min       0.107
25%       0.348
50%       0.413
75%       0.599
max       0.908


MEL union, lesion-conditioned AUC:
                                count  median
family ckpt cam_type     gamma               
act    ha0  finercam     0.6       31   0.611
            gradcam_a    NaN       31   0.579
            gradcam_b    NaN       31   0.376
            map_diff     NaN       31   0.610
       ha3  finercam     0.6       31   0.873
            gradcam_a    NaN       31   0.886
            gradcam_b    NaN       31   0.888
            map_diff     NaN       31   0.642
       ha5  finercam     0.6       31   0.877
                         0.8       31   0.791
            gradcam_a    NaN       31   0.878
            gradcam_b    NaN       31   0.888
            map_diff     NaN       31   0.597
attn   ha0  attncam_a    NaN       31   0.500
            attncam_b    NaN       31   0.492
            attncam_diff Na

In [11]:
# =============================================================================
# How different is the target-class map from the reference-class map.
# EXPECT: near zero everywhere, given the AUC table above.
# =============================================================================
pairs = [("act", "gradcam_a", "gradcam_b"),
         ("attn", "attncam_a", "attncam_b")]

rows = []
for fam, a, b in pairs:
    for ck in ["ha0", "ha3", "ha5"]:
        sa, sb = f"{fam}_{ck}_{a}", f"{fam}_{ck}_{b}"
        vals = []
        for img in eval_df.image_id:
            ca, cb = cams.get((img, sa)), cams.get((img, sb))
            if ca is None or cb is None:
                continue
            na, nb = norm01(ca).ravel(), norm01(cb).ravel()
            ma = topk_fixed(norm01(ca)); mb = topk_fixed(norm01(cb))
            vals.append({
                "pearson": float(np.corrcoef(na, nb)[0, 1]),
                "topk_iou": float((ma & mb).sum() / ((ma | mb).sum() + 1e-8)),
                "mad": float(np.abs(na - nb).mean()),
            })
        v = pd.DataFrame(vals).median()
        rows.append({"family": fam, "ckpt": ck, **v.to_dict()})

disc = pd.DataFrame(rows)
disc.to_csv(OUT_DIR / "class_discriminativeness.csv", index=False)
print("target-class map versus reference-class map, median over images:")
print(disc.round(3).to_string(index=False))

target-class map versus reference-class map, median over images:
family ckpt  pearson  topk_iou   mad
   act  ha0   -0.435     0.000 0.309
   act  ha3    0.993     0.818 0.025
   act  ha5    0.996     0.818 0.019
  attn  ha0   -0.275     0.000 0.193
  attn  ha3    0.960     0.356 0.063
  attn  ha5    0.968     0.333 0.058


In [12]:
# =============================================================================
# Not the analysis. Just enough to see whether anything is there.
# EXPECT E1: cam_auc rising ha0 -> ha3 -> ha5, at least for attention-CAM.
# EXPECT E2: finercam close to gradcam_a. Any large gap is worth a look.
# =============================================================================
mel = long[(long.target == "union_MEL") & long.cam_auc.notna()]

print("E1 preview, target class CAM on MEL union, all 34 images")
for fam, ct in [("attn", "attncam_a"), ("act", "gradcam_a")]:
    s = (mel[(mel.family == fam) & (mel.cam_type == ct)]
         .groupby("ckpt")["cam_auc"].agg(["count", "median"]).round(3))
    print(f"\n  {fam} / {ct}")
    print(s.to_string())

print("\n\nE2 preview, activation ha5, MEL images only")
e2 = mel[(mel.family == "act") & (mel.ckpt == "ha5") & (mel.gt_label == "MEL")]
print(e2.groupby(["cam_type", "gamma"], dropna=False)["cam_auc"]
      .agg(["count", "median"]).round(3).to_string())

print("\n\nby case type, attention ha5:")
print(mel[(mel.family == "attn") & (mel.ckpt == "ha5") &
          (mel.cam_type == "attncam_a")]
      .groupby("case_type")["cam_auc"].agg(["count", "median"]).round(3).to_string())

E1 preview, target class CAM on MEL union, all 34 images

  attn / attncam_a
      count  median
ckpt               
ha0      31   0.507
ha3      31   0.923
ha5      31   0.920

  act / gradcam_a
      count  median
ckpt               
ha0      31   0.555
ha3      31   0.957
ha5      31   0.955


E2 preview, activation ha5, MEL images only
                 count  median
cam_type  gamma               
finercam  0.6       15   0.971
          0.8       15   0.962
gradcam_a NaN       15   0.973
gradcam_b NaN       15   0.972
map_diff  NaN       15   0.730


by case type, attention ha5:
           count  median
case_type               
FN_MEL         3   0.938
FP_MEL        10   0.934
TN_NV          6   0.835
TP_MEL        12   0.922
